# BirdCLEF+ 2026 — Perch v2 + MLP

Perch v2 の 1536次元 Embedding を特徴量として、**MLP（2層NN）** で分類する。
LogisticRegression (LB 0.704) からの改善を目指す。

## Changes from baseline
- LogisticRegression → **MLP (1536→512→206)**
- ReLU + BatchNorm + Dropout(0.3) で過学習を抑制
- PyTorch で学習（GPU 使用、数分で完了）

## Kaggle Settings
| Setting | Value |
|---------|-------|
| Data | `birdclef-2026` (Competition) |
| Models | `google/bird-vocalization-classifier` → `perch_v2` V1 |
| Accelerator | GPU T4 x2 |
| Internet | ON |

In [ ]:
!pip install -q "tensorflow>=2.20" perch-hoplite librosa

In [ ]:
import os, glob, pickle, warnings, time
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'GPU (TF): {tf.config.list_physical_devices("GPU")}')
print(f'GPU (PT): {torch.cuda.is_available()}')

In [ ]:
# ============================================================
# Path Configuration
# ============================================================
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p):
        COMP_DATA = p
        break
assert COMP_DATA, 'Competition data not found'

PERCH_MODEL_PATH = None
saved_models = glob.glob('/kaggle/input/**/saved_model.pb', recursive=True)
for sm in saved_models:
    if 'perch' in sm.lower() or 'bird' in sm.lower():
        PERCH_MODEL_PATH = os.path.dirname(sm)
        break
if PERCH_MODEL_PATH is None and saved_models:
    PERCH_MODEL_PATH = os.path.dirname(saved_models[0])
assert PERCH_MODEL_PATH, 'Perch model not found'

SAMPLE_RATE = 32000
DURATION = 5
WINDOW_SAMPLES = SAMPLE_RATE * DURATION
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Competition data: {COMP_DATA}')
print(f'Perch model: {PERCH_MODEL_PATH}')

In [ ]:
# ============================================================
# Load Perch v2 + Extract embeddings
# ============================================================
def load_perch_model(model_path):
    try:
        from perch_hoplite.zoo import model_configs
        model = model_configs.load_model_by_name('perch_v2', model_path=model_path)
        def embed_fn(waveform):
            emb = np.array(model.embed(waveform).embeddings)
            if emb.ndim > 1: emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
            return emb.astype(np.float32)
        print('Loaded via perch-hoplite')
        return embed_fn
    except Exception as e:
        print(f'perch-hoplite failed: {e}')
    model = tf.saved_model.load(model_path)
    infer_fn = model.signatures['serving_default']
    output_keys = list(infer_fn.structured_outputs.keys())
    embed_key = next((k for k in output_keys if 'embed' in k.lower()), output_keys[0])
    def embed_fn(waveform):
        result = infer_fn(tf.constant(waveform[np.newaxis], dtype=tf.float32))
        emb = result[embed_key].numpy()
        if emb.ndim > 2: emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
        elif emb.ndim == 2: emb = emb[0]
        return emb.astype(np.float32)
    print(f'Loaded via TF SavedModel (key: {embed_key})')
    return embed_fn

embed_fn = load_perch_model(PERCH_MODEL_PATH)
assert embed_fn(np.zeros(WINDOW_SAMPLES, dtype=np.float32)).shape == (1536,)
print('Perch OK')

# Extract
train_df = pd.read_csv(f'{COMP_DATA}/train.csv')
sample_sub = pd.read_csv(f'{COMP_DATA}/sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']
print(f'Training samples: {len(train_df)}')

def load_waveform(path, sr=SAMPLE_RATE, duration=DURATION):
    try: waveform, _ = librosa.load(path, sr=sr, duration=duration)
    except: return np.zeros(sr * duration, dtype=np.float32)
    target = sr * duration
    if len(waveform) < target: waveform = np.pad(waveform, (0, target - len(waveform)))
    return waveform[:target].astype(np.float32)

embeddings = np.zeros((len(train_df), 1536), dtype=np.float32)
for i, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Extracting'):
    embeddings[i] = embed_fn(load_waveform(os.path.join(COMP_DATA, 'train_audio', row['filename'])))
print(f'Embeddings: {embeddings.shape}, NaN: {np.isnan(embeddings).sum()}')

In [ ]:
# ============================================================
# MLP Definition + Training with 5-Fold CV
# ============================================================
class MLPClassifier(nn.Module):
    def __init__(self, input_dim=1536, hidden_dim=512, num_classes=206, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, x): return self.net(x)
    @torch.no_grad()
    def predict_proba(self, x):
        self.eval()
        return torch.softmax(self.forward(x), dim=1).cpu().numpy()

le = LabelEncoder()
y_train = le.fit_transform(train_df['primary_label'].values)
n_classes = len(le.classes_)
print(f'Classes: {n_classes}')

pt_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_tensor = torch.tensor(embeddings, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.long)

HIDDEN_DIM, DROPOUT, LR, EPOCHS, BATCH_SIZE, PATIENCE = 512, 0.3, 1e-3, 30, 256, 5

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(embeddings, y_train)):
    print(f'\n--- Fold {fold} ---', flush=True)
    X_tr = X_tensor[tr_idx].to(pt_device)
    X_va = X_tensor[va_idx].to(pt_device)
    y_tr = y_tensor[tr_idx].to(pt_device)
    y_va = y_tensor[va_idx].to(pt_device)
    
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    model = MLPClassifier(1536, HIDDEN_DIM, n_classes, DROPOUT).to(pt_device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    criterion = nn.CrossEntropyLoss()
    
    best_cmap, patience_cnt, best_state = 0.0, 0, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        
        val_probs = model.predict_proba(X_va)
        aps = []
        for c in range(n_classes):
            y_true_c = (y_va.cpu().numpy() == c).astype(int)
            if y_true_c.sum() > 0:
                aps.append(average_precision_score(y_true_c, val_probs[:, c]))
        cmap = np.mean(aps)
        scheduler.step(avg_loss)
        
        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:02d} | loss={avg_loss:.4f} | cmAP={cmap:.4f}', flush=True)
        
        if cmap > best_cmap:
            best_cmap, patience_cnt = cmap, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}'); break
    
    cv_scores.append(best_cmap)
    print(f'  Fold {fold} Best cmAP: {best_cmap:.4f}', flush=True)

print(f'\nCV Mean cmAP: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})', flush=True)

# Train final model on all data
print('\nTraining final model...', flush=True)
all_loader = DataLoader(TensorDataset(X_tensor.to(pt_device), y_tensor.to(pt_device)),
                        batch_size=BATCH_SIZE, shuffle=True)
final_model = MLPClassifier(1536, HIDDEN_DIM, n_classes, DROPOUT).to(pt_device)
final_opt = torch.optim.Adam(final_model.parameters(), lr=LR, weight_decay=1e-5)
final_crit = nn.CrossEntropyLoss()
for epoch in range(1, EPOCHS + 1):
    final_model.train()
    for xb, yb in all_loader:
        final_opt.zero_grad()
        final_crit(final_model(xb), yb).backward()
        final_opt.step()

torch.save({'model_state_dict': final_model.cpu().state_dict(),
            'hidden_dim': HIDDEN_DIM, 'dropout': DROPOUT, 'n_classes': n_classes},
           f'{OUTPUT_DIR}/perch_v2_mlp.pth')
with open(f'{OUTPUT_DIR}/perch_v2_mlp_le.pkl', 'wb') as f:
    pickle.dump(le, f)
print('Saved MLP model + LabelEncoder', flush=True)

In [ ]:
# ============================================================
# Inference on test soundscapes
# ============================================================
final_model = final_model.to(pt_device).eval()
species_to_idx = {}
for sp in species_cols:
    try: species_to_idx[sp] = int(le.transform([sp])[0])
    except ValueError: pass
print(f'Mapped: {len(species_to_idx)} / {len(species_cols)} species')

test_dir = f'{COMP_DATA}/test_soundscapes'
test_files = sorted(f for f in os.listdir(test_dir) if f.endswith('.ogg'))
print(f'Test files: {len(test_files)}')

rows = []
for fname in tqdm(test_files, desc='Inference'):
    stem = os.path.splitext(fname)[0]
    full_wf, _ = librosa.load(os.path.join(test_dir, fname), sr=SAMPLE_RATE, mono=True)
    if len(full_wf) == 0: continue
    for start in range(0, len(full_wf), WINDOW_SAMPLES):
        window = full_wf[start:start+WINDOW_SAMPLES]
        if len(window) < WINDOW_SAMPLES:
            window = np.pad(window, (0, WINDOW_SAMPLES - len(window)))
        emb = embed_fn(window.astype(np.float32))
        probs = final_model.predict_proba(
            torch.tensor(emb, dtype=torch.float32).unsqueeze(0).to(pt_device))[0]
        bucket_sec = (start // WINDOW_SAMPLES) * DURATION
        row = {'row_id': f'{stem}_{bucket_sec}'}
        for sp in species_cols:
            row[sp] = float(probs[species_to_idx[sp]]) if sp in species_to_idx else 0.0
        rows.append(row)
print(f'Predictions: {len(rows)}')

In [ ]:
# ============================================================
# Build + verify submission
# ============================================================
submission = pd.DataFrame(rows, columns=['row_id'] + species_cols)
submission = submission.set_index('row_id').reindex(sample_sub['row_id'], fill_value=0.0).reset_index()
submission.to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)

assert len(submission) == len(sample_sub)
assert list(submission.columns) == list(sample_sub.columns)
vals = submission.drop('row_id', axis=1).values
print(f'Submission saved')
print(f'  Rows: {len(submission)}, Mean: {vals.mean():.6f}, Max: {vals.max():.4f}, Zero: {(vals==0).mean():.2%}')

In [ ]:
# ============================================================
# Convert Perch to TFLite (for offline submission notebook)
# ============================================================
print(f'Converting: {PERCH_MODEL_PATH}')
converter = tf.lite.TFLiteConverter.from_saved_model(PERCH_MODEL_PATH)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
tflite_model = converter.convert()
with open(f'{OUTPUT_DIR}/perch_v2.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'TFLite: {len(tflite_model)/1024/1024:.1f} MB')

In [ ]:
# Verify all output files
for f in sorted(glob.glob(f'{OUTPUT_DIR}/*')):
    print(f'  {os.path.basename(f):35s} {os.path.getsize(f)/1024:.1f} KB')